# From chat to agents: how a model gets things done
(Part 2 of 3)

In Part 1 we saw that a chat model is a function that continues text, and that a conversation is one long text that we keep sending back.

A model like that can only write. It cannot look anything up, run a calculation or open a file. In this lecture we fix that with a few lines of code around the model, and what we end up with is what people call an **agent**. Coding agents such as Claude Code are built this way.

We keep using the same tiny model. It will get us through the mechanics and then it will run out of ability, which is part of the lesson.

#### Before this lecture

- Run Part 1 once, so that the model is already downloaded.
- Have Claude Code installed ([instructions](https://code.claude.com/docs/en/overview)). To check, open a terminal and type `claude`. If it tells you that you are logged out, type `/login` and follow the instructions. Type `/exit` to leave.

In [ ]:
#!pip install torch "transformers>=4.51" pandas

In [ ]:
import json
import re
import datetime
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

torch.set_grad_enabled(False)               # we only run the model, we never train it
transformers.logging.set_verbosity_error()  # hide the library's warnings

#### Load the chat model from Part 1

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model     = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
streamer  = TextStreamer(tokenizer, skip_prompt=True)

#### A reminder of the chat format from Part 1

Everything we send to the model is one piece of text, made of turns:

```
<|im_start|>ROLE
the text of the turn<|im_end|>
```

| Role | Who it is |
|---|---|
| `system` | the application: instructions for the whole conversation |
| `user` | the person typing |
| `assistant` | the model |

The text always ends with an open assistant turn, `<|im_start|>assistant`, for the model to continue. The empty `<think>` block after it tells this model to answer directly, without writing notes to itself first.

#### And some real data: daily prices for all US stocks in September 2025

In [ ]:
prices = pd.read_csv("../../datasets/market_data/ohlcv_2025-sept.csv.gz")
prices

## Three things a model cannot do

Each prompt below is a one-question conversation, written by hand in the chat format from Part 1.

#### 1. Arithmetic

In [ ]:
prompt = """<|im_start|>user
What is 7847 * 9283?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=120, do_sample=False, streamer=streamer);

#### The real answer

In [ ]:
7847 * 9283

Remember from Part 1 that the model sees these numbers as single digits, and produces its answer one digit at a time, each being the most likely next token. That is not how multiplication works.

#### 2. Anything that happened after it was trained, such as today's date

In [ ]:
prompt = """<|im_start|>user
What is today's date?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=80, do_sample=False, streamer=streamer);

#### 3. Facts it has never seen, such as a price in our file

In [ ]:
prompt = """<|im_start|>user
What was the closing price of AAPL on 2025-09-10?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=80, do_sample=False, streamer=streamer);

#### What was it really?

In [ ]:
prices[(prices.ticker == "AAPL") & (prices.date == "2025-09-10")]

Look at how the wrong answer was delivered: a precise number, stated with confidence. The model is doing what it always does, producing likely text. A price that looks right is likely text.

People call this **hallucination**. Bigger models do it less often. They still do it.

## The first fix: put what the model needs into the prompt

#### Chat products know today's date because somebody writes it into the system prompt

In [ ]:
today = datetime.date.today()

prompt = f"""<|im_start|>system
Today's date is {today}.<|im_end|>
<|im_start|>user
What is today's date?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
print(prompt)

In [ ]:
ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=80, do_sample=False, streamer=streamer);

#### The same works for data: paste it into the question

In [ ]:
table = prices.loc[prices.ticker == "AAPL", ["date", "close"]].head(8).to_string(index=False)
print(table)

In [ ]:
prompt = f"""<|im_start|>user
Here are closing prices for AAPL:

{table}

What was the closing price of AAPL on 2025-09-10?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
print(prompt)

In [ ]:
ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=80, do_sample=False, streamer=streamer);

Correct this time, because the answer is in the text it is continuing.

When a program finds the relevant documents or rows and pastes them into the prompt for you, it is called **retrieval-augmented generation**, or RAG. "Chat with your documents" products work this way. The paper that named it: [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) (2020).

#### Why not paste in the whole file? Count the tokens for one ticker, one column, one month

In [ ]:
one_ticker = prices.loc[prices.ticker == "AAPL", ["date", "close"]].to_string(index=False)
len(tokenizer.encode(one_ticker))

In [ ]:
prices.ticker.nunique()   # and there are this many tickers

#### This is the most text our model can take in at once (its context window), in tokens

In [ ]:
model.config.max_position_embeddings

Commercial models have much larger context windows, but real data is much larger again. Somebody has to decide which small part of the data goes into the prompt. Ideally, the model decides.

## The second fix: let the model ask for what it needs

The idea: tell the model, in the system prompt, that some functions exist. When it wants one, it writes the call as text. Our code runs the function and pastes the result back into the conversation.

Start with the simplest possible function.

In [ ]:
def multiply(a, b):
    return a * b

multiply(7847, 9283)

#### Describe the function as data: its name, what it does, and its arguments

In [ ]:
multiply_tool = {
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
    },
}

#### Put the description into a system prompt, with instructions for how to ask

In [ ]:
system_prompt = f"""# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{json.dumps(multiply_tool)}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{{"name": <function-name>, "arguments": <args-json-object>}}
</tool_call>"""

print(system_prompt)
print(prompt)

Read it. It is plain English plus a description of our function. This exact wording is the one this model family saw during training, in the same way it saw `<|im_start|>`.

#### Ask the arithmetic question again, with that system prompt in front

In [ ]:
prompt = f"""<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
What is 7847 * 9283?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
print(prompt)

In [ ]:
ids = tokenizer.encode(prompt, return_tensors="pt")
output = model.generate(ids, max_new_tokens=80, do_sample=False, streamer=streamer)

The model did not answer. It wrote a request, and then stopped.

Nothing has been multiplied. The model cannot run anything: all it produced is text that says which function it wants and with which arguments. Running it is our job.

#### Pull the request out of the model's reply

In [ ]:
reply = tokenizer.decode(output[0][len(ids[0]):])
call  = json.loads(reply.split("<tool_call>")[1].split("</tool_call>")[0])
call

#### Run the function it asked for

In [ ]:
result = multiply(**call["arguments"])
result

#### Add the model's request and our result to the conversation

In [ ]:
prompt += reply + f"""
<|im_start|>user
<tool_response>
{result}
</tool_response><|im_end|>
<|im_start|>assistant
<think>

</think>

"""
print(prompt)

#### ...and let the model continue

In [ ]:
ids = tokenizer.encode(prompt, return_tensors="pt")
model.generate(ids, max_new_tokens=80, do_sample=False, streamer=streamer);

The right answer, from a model that cannot multiply. It did not need to: it needed to know that a function exists and how to ask for it.

This is called **tool calling** or **function calling**, and the functions are called **tools**.

## More tools: another piece of arithmetic, and a lookup in our data

In [ ]:
def add(a, b):
    return a + b

add_tool = {
    "type": "function",
    "function": {
        "name": "add",
        "description": "Add two numbers.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
    },
}

In [ ]:
def get_close(ticker, date):
    row = prices[(prices.ticker == ticker) & (prices.date == date)]
    if len(row) == 0:
        return f"no data for {ticker} on {date}"
    return round(float(row.close.iloc[0]), 2)

get_close("AAPL", "2025-09-10")

In [ ]:
get_close_tool = {
    "type": "function",
    "function": {
        "name": "get_close",
        "description": "Get the closing price of a stock on a date.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Ticker symbol, for example AAPL"},
                "date":   {"type": "string", "description": "Date as YYYY-MM-DD"},
            },
            "required": ["ticker", "date"],
        },
    },
}

#### The system prompt now lists all three

In [ ]:
system_prompt = f"""# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{json.dumps(multiply_tool)}
{json.dumps(add_tool)}
{json.dumps(get_close_tool)}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{{"name": <function-name>, "arguments": <args-json-object>}}
</tool_call>"""

print(system_prompt)
print(prompt)

In [ ]:
tools = {"multiply": multiply, "add": add, "get_close": get_close}

## Put it in a loop: that is an agent

We did four things by hand: let the model write, look for a tool call, run it, paste the result back. Now we repeat them until the model replies without asking for anything.

In [ ]:
def agent(question):
    conversation = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n"

    for step in range(5):
        conversation += "<|im_start|>assistant\n<think>\n\n</think>\n\n"
        ids    = tokenizer.encode(conversation, return_tensors="pt")
        output = model.generate(ids, max_new_tokens=200, do_sample=False, streamer=streamer)
        reply  = tokenizer.decode(output[0][len(ids[0]):])
        conversation += reply + "\n"

        calls = re.findall("<tool_call>\n(.*?)\n</tool_call>", reply)
        if len(calls) == 0:
            break                                  # no request: the model is finished

        results = []
        for call in calls:
            call   = json.loads(call)
            result = tools[call["name"]](**call["arguments"])
            print(">>> we ran", call["name"], call["arguments"], "and got", result)
            results.append(f"<tool_response>\n{result}\n</tool_response>")
        conversation += "<|im_start|>user\n" + "\n".join(results) + "<|im_end|>\n"

    return conversation

#### The multiplication again. This time the model has three tools to choose from

In [ ]:
conversation = agent("What is 7847 * 9283?")

#### The whole text of that conversation: our system prompt, the question, the model's request, our result, the model's answer

In [ ]:
print(conversation)

#### The price the model made up earlier

In [ ]:
conversation = agent("What was the closing price of AAPL on 2025-09-10?")

#### A question that needs two lookups

In [ ]:
conversation = agent("Which closed higher on 2025-09-10, AAPL or MSFT?")

#### A question that needs no tool at all

In [ ]:
conversation = agent("In one sentence, why do bond prices fall when interest rates rise?")

Nobody told the model which tool to use, how many times, or whether to use one at all. It chose. The loop only does what it is asked and reports back.

A model in a loop, with tools, deciding for itself what to do next until the job is done: that is an **agent**. The loop above is the whole idea.

Our three tools are deliberately tiny. A tool can be any function: run a database query, send an email, place an order, run a program. The more general the tool, the more the model can do with it, and the more can go wrong.

Some history, because you will meet these names:
- **ReAct** (2022) is the paper that made this pattern popular: the model alternates between reasoning in text and acting through tools. [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- In 2023 the model providers built it into their APIs as **function calling** or **tool use**: you pass a list of tool descriptions like our dictionaries, and get the requests back as data rather than text. [Anthropic](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview), [OpenAI](https://platform.openai.com/docs/guides/function-calling)
- **MCP** (Model Context Protocol, 2024) is a standard way to package a set of tools so that any agent can use them. We use it in Part 3. [modelcontextprotocol.io](https://modelcontextprotocol.io)

#### Problem set

Turn `agent` into a program you can talk to in a terminal, the way `2_chat_without_memory.py` is a program for a plain chat.

#### The library we are using does what we did by hand: give it the descriptions and it writes the same system prompt

In [ ]:
messages = [{"role": "user", "content": "What is 7847 * 9283?"}]
by_library = tokenizer.apply_chat_template(messages, tools=[multiply_tool, add_tool, get_close_tool], tokenize=False)

by_hand = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\nWhat is 7847 * 9283?<|im_end|>\n"

by_library == by_hand

## Where a small model runs out

#### A question that needs two lookups and then some arithmetic

In [ ]:
conversation = agent("I own 150 shares of AAPL and 40 shares of MSFT. What was my portfolio worth at the close on 2025-09-10?")

In [ ]:
150 * get_close("AAPL", "2025-09-10") + 40 * get_close("MSFT", "2025-09-10")

It does not get there. Read what it wrote. Did it look up both prices? Did it multiply and add? Every tool it needed was available.

#### A question our tools do not fit well

In [ ]:
conversation = agent("On which day in September 2025 did AAPL close highest?")

#### The real answer

In [ ]:
prices[prices.ticker == "AAPL"].sort_values("close").tail(1)

To answer this with `get_close`, the model would have to ask for every trading day in the month and compare. It did not do that. If it gave you a date anyway, where did that date come from?

Nothing is wrong with the loop. The loop is trivial, and it is the same loop in every agent you will use. What differs is the model inside it: how well it plans, whether it notices it is not finished, what it does when a tool returns an error. That is where the capability is, and it is why the same twenty lines are a toy with this model and a product with a large one.

The failures are worth remembering, because large models have the same ones, less often: not using a tool they have, doing in their head what they should have calculated, stopping early, giving up, and reporting any of it in a confident sentence.

## The same loop with a strong model: a coding agent

Claude Code is this loop with a large model and a different set of tools: read a file, edit a file, run a command in the terminal, search the web. Because one of its tools runs commands, it can write a program and run it, which makes it far more general than our two functions.

We can watch its loop from here. First save a small file for it to work with:

In [ ]:
prices.loc[prices.ticker.isin(["AAPL", "MSFT"]), ["ticker", "date", "close"]].to_csv("prices.csv", index=False)

#### Ask Claude Code the portfolio question, and ask it to report every step as data

`-p` means: answer this one prompt and exit. The other two options make it print each step of its loop as a line of JSON.

(If the output below talks about authentication, you are logged out: open a terminal, type `claude`, then `/login`.)

In [ ]:
trace = !claude -p "Read prices.csv. I own 150 shares of AAPL and 40 shares of MSFT. What was my portfolio worth at the close on 2025-09-10?" --output-format stream-json --verbose
len(trace)

#### Each line is one event. Print who is speaking and what kind of thing they said

In [ ]:
for line in trace:
    if not line.startswith("{"):
        continue
    event = json.loads(line)
    if event["type"] in ("assistant", "user"):
        for block in event["message"]["content"]:
            print(event["type"], "|", block["type"], block.get("name", ""), "|", str(block.get("text") or block.get("input") or block.get("content") or "")[:300])

Compare with our loop:

| In the trace | In our `agent` function |
|---|---|
| `assistant` / `tool_use`, with a tool name (`Read`) and arguments | the model writes `<tool_call>` |
| `user` / `tool_result` | we paste `<tool_response>` back into the conversation |
| `assistant` / `thinking` | the notes the model writes to itself (Part 1). Claude Code does not print them here |
| `assistant` / `text` at the end | the reply with no tool call, which ends the loop |

Notice that the tool result comes back as a `user` message, exactly as ours did. To the model, everything it did not write itself is somebody else's turn.

#### And check its answer. We know how to compute this one

In [ ]:
150 * get_close("AAPL", "2025-09-10") + 40 * get_close("MSFT", "2025-09-10")

## What to take away

- A model on its own cannot calculate reliably, does not know anything after its training, and will invent facts it has not seen, confidently.
- Whatever is in the prompt, it can use. Getting the right things into the prompt is most of the craft.
- A tool is a function plus a description of it in the system prompt. The model writes a request as text, and our code runs it and pastes the result back.
- An agent is a model in a loop with tools, deciding what to do next until it is done.
- The loop is simple and the same everywhere. The quality of an agent comes from the model, the tools it is given, and what is put in its prompt.
- An agent's answer is still generated text. Check it.

In Part 3 we work with Claude Code directly: giving it context, teaching it a skill, and connecting it to outside data.

## Things to try at home

- Ask `agent` your own questions about the price file, and `print` the conversation it returns. Which ones work? When it fails, read what it wrote: where exactly did it go wrong?
- Write another tool, for example `subtract(a, b)` or `get_volume(ticker, date)`, describe it, and add it to the system prompt and to `tools`.
- Change the order in which the tools are listed in the system prompt and ask the same questions again. With a model this small, the answers change. What does that tell you?
- Change a tool's description so that it is vague or misleading. Does the model still use it correctly?
- Make `get_close` return a wrong number on purpose. Does the model notice?
- Give `agent` a question with a ticker that does not exist. What does the model do with the "no data" reply?
- Run the Claude Code cell with a harder question and read the trace. How many tool calls did it make?